# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [8]:
#Import necessary libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import anthropic
from IPython.display import Markdown, display, update_display
import socket

In [2]:
#load the .env file
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

In [3]:
system_message = 'You are a helpful assistant.  If you do not know the answer to a question, please say so.  Respond in Markdown.'

openai = OpenAI()
claude = anthropic.Anthropic()
history = []

In [4]:
def chat_openai(message, history):
    messages = [{"role": "system", "content": system_message}]
    messages.extend(history)
    messages.append({"role": "user", "content": message})

    stream = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
        stream=True
    )

    response = ""

    # Generator function to yield content and update history
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        response += delta
        yield response  # stream to Gradio frontend

    # Important: update history *after* the full response is streamed
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

In [5]:
def chat_claude(message, history):
    result = claude.messages.stream(
        model="claude-3-5-haiku-20241022",
        max_tokens = 1000,
        temperature = 0.4,
        system = system_message,
        messages = history + [{"role":"user","content":message}]
    )
    response = ""
    with result as stream:
        for chunk in stream.text_stream:
            response += chunk
            yield response

    history.append({"role":"user", "content":message})
    history.append({"role":"assistant", "content":response})

In [6]:
def stream_chat(message, model):
    if model == 'OpenAI':
        yield from chat_openai(message, history)
    if model == 'Claude':
        yield from chat_claude(message, history)
    if not(model == 'OpenAI' or model == 'Claude'):
        raise ValueError('Sorry we don''t support that model yet!')

In [7]:
view = gr.Interface(
    fn=stream_chat,
    inputs=[gr.Textbox(label="Message:"), gr.Dropdown(["OpenAI","Claude"], label="Model:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()
#gr.ChatInterface(fn=chat_openai, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
compIpAdress = socket.gethostbyname(socket.gethostname())
# Print the IP address of the computer.
print("The IP Address of this computer is [", compIpAdress, ']')

The IP Address of this computer is [ 192.168.30.190 ]
